# Data Leak because Preprocessing before Split DataSet

### 1️⃣ Import Library

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

### 2️⃣ Data Dummy

In [2]:
np.random.seed(42)

n = 500
df = pd.DataFrame({
    'age': np.random.randint(18, 70, n),
    'income': np.random.normal(50000, 15000, n),
    'savings': np.random.normal(20000, 8000, n)
})

In [3]:
# Target: default jika income & savings kecil, atau usia muda
df['default'] = ((df['age'] < 30) & (df['income'] < 40000)) | (df['savings'] < 10000)
df['default'] = df['default'].astype(int)

In [4]:
X = df.drop(columns='default')
y = df['default']

### 3️⃣ ❌ Kesalahan Umum: Scaling sebelum Split

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # fit dilakukan di seluruh data (train + test!)

X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

In [17]:
X_train_bad

array([[-1.27953869, -0.68312517,  0.05756403],
       [ 0.38479363,  0.12808506,  0.53718199],
       [-1.2129654 ,  0.339235  ,  0.56042481],
       ...,
       [ 1.5831129 ,  0.79219881,  0.88088288],
       [ 1.25024644, -0.34785265, -0.35701364],
       [-0.14779271,  0.01054298,  1.3086479 ]], shape=(350, 3))

In [6]:
model_bad = LogisticRegression()
model_bad.fit(X_train_bad, y_train_bad)

LogisticRegression()

In [7]:
train_pred_bad = model_bad.predict(X_train_bad)
test_pred_bad = model_bad.predict(X_test_bad)

In [9]:
print("❌ MODEL DENGAN LEAKAGE (scaling before split)")
print("Train Accuracy :", accuracy_score(y_train_bad, train_pred_bad))
print("Test Accuracy  :", accuracy_score(y_test_bad, test_pred_bad))

❌ MODEL DENGAN LEAKAGE (scaling before split)
Train Accuracy : 0.88
Test Accuracy  : 0.8733333333333333


### 4️⃣ ✅ Cara Benar: Split dulu, baru Scaling

In [10]:
X_train_good, X_test_good, y_train_good, y_test_good = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [18]:
X_train_good

,age,income,savings
5,25,39384.958016,21478.688990
116,50,51465.141478,25268.354181
45,26,54609.492813,25452.005579
16,20,43399.332700,14984.263538
462,61,49435.479464,17640.402574
...,...,...,...
106,18,54840.778405,15582.215646
270,24,40060.643616,19805.163262
348,68,61354.829250,27984.080879
435,63,44377.687887,18202.934800


In [11]:
scaler2 = StandardScaler()
X_train_scaled = scaler2.fit_transform(X_train_good)  # fit hanya di training data
X_test_scaled = scaler2.transform(X_test_good)        # transform test pakai scaler yang sama

In [12]:
model_good = LogisticRegression()
model_good.fit(X_train_scaled, y_train_good)


LogisticRegression()

In [13]:
train_pred_good = model_good.predict(X_train_scaled)
test_pred_good = model_good.predict(X_test_scaled)

In [14]:
print("\n✅ MODEL TANPA LEAKAGE (split sebelum scaling)")
print("Train Accuracy :", accuracy_score(y_train_good, train_pred_good))
print("Test Accuracy  :", accuracy_score(y_test_good, test_pred_good))


✅ MODEL TANPA LEAKAGE (split sebelum scaling)
Train Accuracy : 0.88
Test Accuracy  : 0.8733333333333333


### 5️⃣ Perbandingan Hasil

In [15]:
print("\n📊 Perbandingan Akurasi")
comparison = pd.DataFrame({
    'Scenario': ['Dengan Leakage (fit di seluruh data)', 'Tanpa Leakage (fit hanya di train)'],
    'Train Accuracy': [accuracy_score(y_train_bad, train_pred_bad),
                       accuracy_score(y_train_good, train_pred_good)],
    'Test Accuracy': [accuracy_score(y_test_bad, test_pred_bad),
                      accuracy_score(y_test_good, test_pred_good)]
})


📊 Perbandingan Akurasi


In [16]:
print(comparison)

                               Scenario  Train Accuracy  Test Accuracy
0  Dengan Leakage (fit di seluruh data)            0.88       0.873333
1    Tanpa Leakage (fit hanya di train)            0.88       0.873333
